# What is RAG?

Ans :- Retrieval-Augmented Generation (RAG) in Python is a technique to enhance Large Language Models (LLMs) by connecting them to external, up-to-date knowledge bases to generate more accurate and contextually relevant responses.

# How RAG Works in Python
Ans :- Building a RAG system in Python typically involves these core stages:

    1) Loading Data
    2) Indexing and Embedding
    3) Storing
    4) Retrieval
    5) Generation

In [1]:
#!pip install langchain-community

#!pip install sentence-transformers

#! pip install sentence-transformers

# ! pip install faiss-cpu

#!pip install pypdf

#!pip install reportlab

In [2]:
import os
os.environ['HF_TOKEN']="add your huggingface token here"

In [3]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import ChatHuggingFace


# -----------------------------
# 1️⃣ Create Sample Documents
# -----------------------------
documents = [
    Document(page_content="Python is a programming language."),
    Document(page_content="Machine learning allows computers to learn from data."),
    Document(page_content="Hugging Face provides open-source AI tools.")
]

# -----------------------------
# 2️⃣ Create Embeddings Model
# -----------------------------
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# -----------------------------
# 3️⃣ Create Vector Store
# -----------------------------
vectorstore = FAISS.from_documents(documents, embeddings)

retriever = vectorstore.as_retriever()

# -----------------------------
# 4️⃣ Load LLM from Hugging Face
# -----------------------------
hf_llm = HuggingFaceEndpoint(
    repo_id="moonshotai/Kimi-K2-Instruct-0905",
    huggingfacehub_api_token=os.environ["HF_TOKEN"],
    task="conversational",
    temperature=0.3,
    max_new_tokens=300,
)

llm = ChatHuggingFace(llm=hf_llm)

# -----------------------------
# 5️⃣ Create RAG Prompt
# -----------------------------
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.

Context:
{context}

Question:
{question}
""")

# -----------------------------
# 6️⃣ RAG Chain
# -----------------------------
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)

# -----------------------------
# 7️⃣ Ask Question
# -----------------------------
response = rag_chain.invoke("What is java?")

print(response)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The context does not mention Java.


In [4]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint, ChatHuggingFace
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# -----------------------------
# 1️⃣ Load PDF (Dynamic Document)
# -----------------------------
pdf_path = "/Users/upforcetech/Downloads/apj-abdul-kalam-biography.pdf"   # <-- change to your PDF file path

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Loaded {len(documents)} pages from PDF")


# -----------------------------
# 2️⃣ Split into Chunks
# -----------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

split_docs = text_splitter.split_documents(documents)

print(f"Created {len(split_docs)} text chunks")


# -----------------------------
# 3️⃣ Create Embeddings
# -----------------------------
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# -----------------------------
# 4️⃣ Create Vector Store
# -----------------------------
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever()


# -----------------------------
# 5️⃣ Load Chat Model (Groq-backed)
# -----------------------------
hf_llm = HuggingFaceEndpoint(
    repo_id="moonshotai/Kimi-K2-Instruct-0905",
    huggingfacehub_api_token=os.environ["HF_TOKEN"],
    task="conversational",
    temperature=0.3,
    max_new_tokens=300,
)

llm = ChatHuggingFace(llm=hf_llm)


# -----------------------------
# 6️⃣ Create RAG Prompt
# -----------------------------
prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the context below.
If the answer is not in the context, say:
"I don't know based on the provided document."

Context:
{context}

Question:
{question}
""")


# -----------------------------
# 7️⃣ Format Retrieved Docs
# -----------------------------
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# -----------------------------
# 8️⃣ RAG Chain
# -----------------------------
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)


# -----------------------------
# 9️⃣ Ask Question
# -----------------------------
query = "What is the main topic of this document?"
response = rag_chain.invoke(query)

print("\nAnswer:\n", response)

Loaded 6 pages from PDF
Created 15 text chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Answer:
 The main topic of this document is the biography of Dr. APJ Abdul Kalam, highlighting his life, achievements, writings, and legacy.


In [11]:
import os
import json
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS


# -----------------------------
# 1️⃣ Load PDF
# -----------------------------
loader = PyPDFLoader("/Users/upforcetech/Downloads/apj-abdul-kalam-biography.pdf")   # Put your PDF file here
documents = loader.load()


# -----------------------------
# 2️⃣ Split into Chunks
# -----------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)


# -----------------------------
# 3️⃣ Create Embeddings
# -----------------------------
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()


# -----------------------------
# 4️⃣ Load LLM
# -----------------------------
hf_llm = HuggingFaceEndpoint(
    repo_id="moonshotai/Kimi-K2-Instruct-0905",
    huggingfacehub_api_token=os.environ["HF_TOKEN"],
    task="conversational",
    temperature=0.7,
    max_new_tokens=900
)

llm = ChatHuggingFace(llm=hf_llm)


# -----------------------------
# 5️⃣ MCQ Prompt
# -----------------------------
prompt = ChatPromptTemplate.from_template("""
You are an expert teacher.

Based ONLY on the following context, generate {num_questions} multiple choice questions.

Context:
{context}

Return output strictly in JSON format:

[
  {{
    "question": "Question text",
    "options": ["A", "B", "C", "D"],
    "answer": "Correct option text"
  }}
]
""")


# -----------------------------
# 6️⃣ Helper to format docs
# -----------------------------
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# -----------------------------
# 7️⃣ RAG MCQ Chain
# -----------------------------
rag_chain = (
    {
        "context": lambda x: format_docs(
            retriever.invoke(x["question"])
        ),
        "num_questions": lambda x: x["num_questions"]
    }
    | prompt
    | llm
    | StrOutputParser()
)


# -----------------------------
# 8️⃣ Generate MCQs
# -----------------------------
response = rag_chain.invoke({
    "question": "Biography of APJ Abdul Kalam",
    "num_questions": 10
})

print(response)

mcqs = json.loads(response)

# Pretty print
for i, q in enumerate(mcqs, 1):
    print(f"\nQ{i}: {q['question']}")
    for idx, option in enumerate(q["options"]):
        print(f"   {chr(65+idx)}. {option}")
    print(f"Answer: {q['answer']}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[
  {
    "question": "Which book written by Dr. APJ Abdul Kalam primarily focused on making India a superpower?",
    "options": ["Wings of Fire", "India 2020", "Mission India", "The Luminous Sparks"],
    "answer": "India 2020"
  },
  {
    "question": "How many people approximately attended Dr. Kalam’s funeral?",
    "options": ["25,000", "30,000", "35,000", "40,000"],
    "answer": "35,000"
  },
  {
    "question": "What was Dr. Kalam’s role in India’s defence programs?",
    "options": ["Army Chief", "Naval Commander", "Key figure in missile & nuclear programs", "Air Marshal"],
    "answer": "Key figure in missile & nuclear programs"
  },
  {
    "question": "Which award did Dr. Kalam receive in the year 2000?",
    "options": ["Padma Bhushan", "Ramanujan Award", "Bharat Ratna", "Padma Vibhushan"],
    "answer": "Ramanujan Award"
  },
  {
    "question": "In which city did Dr. Kalam pass away on 27 July 2015?",
    "options": ["Rameswaram", "New Delhi", "Shillong", "Chennai"],
   

In [9]:
import os
import json
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
#from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet


# -----------------------------
# 1️⃣ Load PDF
# -----------------------------
loader = PyPDFLoader("/Users/upforcetech/Downloads/apj-abdul-kalam-biography.pdf")   # Put your PDF file here
documents = loader.load()


# -----------------------------
# 2️⃣ Split into Chunks
# -----------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)


# -----------------------------
# 3️⃣ Create Embeddings
# -----------------------------
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()


# -----------------------------
# 4️⃣ Load LLM
# -----------------------------
hf_llm = HuggingFaceEndpoint(
    repo_id="moonshotai/Kimi-K2-Instruct-0905",
    huggingfacehub_api_token=os.environ["HF_TOKEN"],
    task="conversational",
    temperature=0.7,
    max_new_tokens=900
)

llm = ChatHuggingFace(llm=hf_llm)


# -----------------------------
# 5️⃣ MCQ Prompt
# -----------------------------
prompt = ChatPromptTemplate.from_template("""
You are an expert teacher.

Based ONLY on the following context, generate {num_questions} multiple choice questions.

Context:
{context}

Return output strictly in JSON format:

[
  {{
    "question": "Question text",
    "options": ["A", "B", "C", "D"],
    "answer": "Correct option text"
  }}
]
""")


# -----------------------------
# 6️⃣ Helper to format docs
# -----------------------------
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# -----------------------------
# 7️⃣ RAG MCQ Chain
# -----------------------------

json_parser = JsonOutputParser()
rag_chain = (
    {
        "context": lambda x: format_docs(
            retriever.invoke(x["question"])
        ),
        "num_questions": lambda x: x["num_questions"]
    }
    | prompt
    | llm
    | json_parser
)


# -----------------------------
# 8️⃣ Generate MCQs
# -----------------------------
response = rag_chain.invoke({
    "question": "Biography of APJ Abdul Kalam",
    "num_questions": 12
})

print(response)

mcqs = response


# -----------------------------
# 7️⃣ Create MCQ PDF
# -----------------------------
pdf = SimpleDocTemplate("mcq_output.pdf")
elements = []
styles = getSampleStyleSheet()

for i, q in enumerate(mcqs, 1):
    elements.append(Paragraph(f"<b>Q{i}. {q['question']}</b>", styles["Normal"]))
    elements.append(Spacer(1, 0.2 * inch))
    
    for option in q["options"]:
        elements.append(Paragraph(option, styles["Normal"]))
        elements.append(Spacer(1, 0.1 * inch))
    
    elements.append(Paragraph(f"<b>Answer:</b> {q['answer']}", styles["Normal"]))
    elements.append(Spacer(1, 0.5 * inch))

pdf.build(elements)

print("MCQ PDF generated successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'question': 'Which town was Dr. APJ Abdul Kalam’s body taken to for his funeral?', 'options': ['New Delhi', 'Rameswaram', 'Shillong', 'Chennai'], 'answer': 'Rameswaram'}, {'question': 'Approximately how many people attended Dr. Kalam’s funeral?', 'options': ['15 000', '25 000', '35 000', '45 000'], 'answer': '35 000'}, {'question': 'In which year did Dr. Kalam receive the Ramanujan Award?', 'options': ['1998', '2000', '2002', '2004'], 'answer': '2000'}, {'question': 'Which book by Dr. Kalam outlines strategies to make India a superpower?', 'options': ['Wings of Fire', 'India 2020', 'Mission India', 'The Luminous Sparks'], 'answer': 'India 2020'}, {'question': 'What was Dr. Kalam’s position in his family among his siblings?', 'options': ['Eldest', 'Second', 'Middle', 'Youngest'], 'answer': 'Youngest'}, {'question': 'On what date was Dr. APJ Abdul Kalam born?', 'options': ['15 October 1931', '15 October 1935', '27 July 1931', '27 July 1935'], 'answer': '15 October 1931'}, {'question': 